In [18]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict
from dotenv import load_dotenv

In [19]:
load_dotenv()
model = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.0,
)

In [20]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    evaluation : int # Home work

In [21]:
def create_outline(state: BlogState) -> BlogState:

    # fetch title
    title = state['title']
    # call llm gen outline
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state
    state['outline'] = outline
    return state

In [22]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline'] # current outline

    prompt = f'write a detailed blog on the title - {title} using the following outline \n {outline}'
    content = model.invoke(prompt).content

    state['content'] = content
    return state

In [23]:
def blog_evaluate(state: BlogState) -> BlogState:
    outline = state['outline']
    content = state['content']

    prompt = f'Based on this outline {outline} Rate my blog {content} and generate integer score: '
    evaluation = model.invoke(prompt)

    state['evaluation'] = evaluation
    return state

In [25]:
graph = StateGraph(BlogState)

# nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('blog_evaluate',blog_evaluate)

# edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'blog_evaluate')
graph.add_edge('blog_evaluate', END)
workflow = graph.compile()

In [26]:
intial_state = {'title': 'rise of ai in india'}
final_state = workflow.invoke(intial_state)
print(final_state)

{'title': 'rise of ai in india', 'outline': '**Blog Title (working)**  \n*The Rise of AI in India: Drivers, Impact, Challenges & the Road Ahead*  \n\n---\n\n## 1. Introduction  \n- **Hook:** A striking statistic or anecdote (e.g., “India is projected to host 20\u202f% of the world’s AI talent by 2030”).  \n- **Why it matters:** AI as a catalyst for economic growth, job creation, and social transformation in a country of 1.4\u202fbn people.  \n- **Scope of the post:** Overview of the ecosystem (policy, talent, startups, industry adoption, challenges) and what readers can expect to learn.  \n\n---\n\n## 2. Historical Context – From Early Experiments to a National Priority  \n### 2.1 Early AI research in Indian academia (1970s‑1990s)  \n- IITs, IISc, ISI Kolkata – pioneering work in pattern recognition, expert systems.  \n- Limited computing resources & isolated research clusters.  \n\n### 2.2 The turn of the millennium – Global AI boom & Indian IT’s role  \n- Outsourcing of AI‑related se

In [27]:
print(final_state['outline'])

**Blog Title (working)**  
*The Rise of AI in India: Drivers, Impact, Challenges & the Road Ahead*  

---

## 1. Introduction  
- **Hook:** A striking statistic or anecdote (e.g., “India is projected to host 20 % of the world’s AI talent by 2030”).  
- **Why it matters:** AI as a catalyst for economic growth, job creation, and social transformation in a country of 1.4 bn people.  
- **Scope of the post:** Overview of the ecosystem (policy, talent, startups, industry adoption, challenges) and what readers can expect to learn.  

---

## 2. Historical Context – From Early Experiments to a National Priority  
### 2.1 Early AI research in Indian academia (1970s‑1990s)  
- IITs, IISc, ISI Kolkata – pioneering work in pattern recognition, expert systems.  
- Limited computing resources & isolated research clusters.  

### 2.2 The turn of the millennium – Global AI boom & Indian IT’s role  
- Outsourcing of AI‑related services (e.g., IBM, Accenture).  
- Emergence of R&D labs in multinational

In [28]:
print(final_state['content'])

**The Rise of AI in India: Drivers, Impact, Challenges & the Road Ahead**  
*By [Your Name] – [Date]*  

---

## 1. Introduction  

### Hook  
> **“India is projected to host 20 % of the world’s AI talent by 2030.”**  
> — *World Economic Forum, 2023*  

In a nation of **1.4 billion people**, that statistic is more than a brag‑sheet line—it’s a signal that artificial intelligence (AI) is poised to become a cornerstone of India’s economic engine, a catalyst for job creation, and a lever for social transformation.

### Why it matters  
- **Economic growth:** AI could add **US$ 1 trillion** to India’s GDP by 2030 (NITI Aayog).  
- **Job creation:** While automation will reshape roles, AI‑enabled services are expected to generate **10‑12 million** new jobs across tech, healthcare, agriculture, and manufacturing.  
- **Social impact:** From early‑cancer detection in tier‑2 cities to precision farming for smallholder growers, AI is already reaching the “last mile.”

### What you’ll learn  
T

In [35]:
print(final_state['evaluation'])

content='**Overall Rating: 8.5\u202f/\u202f10**\n\n| Aspect | Score | Comments |\n|--------|------|----------|\n| **Structure & Flow** | 9 | The outline follows a logical, reader‑friendly progression—from history to drivers, ecosystem, impact, policy, challenges, and future trends. The “Practical Takeaways” section nicely bridges the gap between insight and action. |\n| **Depth & Breadth** | 9 | You cover the major pillars of India’s AI story (government, academia, corporates, startups) and back each claim with concrete data points or examples. The inclusion of sector‑specific startup tables and success‑story snapshots adds credibility. |\n| **Data & Evidence** | 8 | Numbers (e.g., 750\u202fM internet users, 30\u202f% of global AI papers) are recent and sourced. A few citations (World Economic Forum, NITI Aayog, IDC) would strengthen the factual backbone and help SEO. |\n| **Readability & Engagement** | 8 | The mix of tables, bullet points, and suggested visual assets (infographics, in

In [33]:
import re

text = final_state['evaluation'].content  # extract the string
score = re.search(r'Overall Rating:\s*(\d+\.?\d*)', text).group(1)
print(score)  # 8.5   

8.5
